# Base DINOv3 Visual Nearest-Neighbor Search

Cleaned/deduplicated version of the original notebook. It had grown two near-identical copies of the setup/model-loading/embedding-helper code (a "first run" section and a "later sessions" section that redefined everything again) — consolidated into one setup you run once per Colab runtime.

**Bug fixed during cleanup:** the old "later sessions" section reloaded the *full* saved index into the same variable (`database_embeddings`/`database_records`) that the held-out 80/20 evaluation split used earlier in the notebook, silently overwriting the gallery-only split with a gallery that also contained the test queries themselves — so Recall@K was being computed with the answer already sitting in the gallery. This version keeps the full index (`full_index_*`) and the eval-only gallery/query split (`gallery_*` / `heldout_query_*`) as separate variables so they can't collide. Also fixed: `show_search_results` is now defined before its first call (previously called one cell before its definition).

1. Load the dataset, build normalized DINOv3 image embeddings, save the reusable index.
2. Reload the index in later sessions without rebuilding it.
3. Interactive nearest-neighbor search for a query image + product-level aggregation.
4. Held-out 80/20 evaluation (R@1/R@5/R@10/MRR/median/mean rank).

The initial index is exact cosine search over a matrix — fine for this catalog size, swap for FAISS/HNSW later without regenerating embeddings.

**Repointed at the current dataset (2026-07-27):** `DATASET_ROOT` now points at `/content/drive/MyDrive/apparel_dataset` (1115 records, 6 brands), read from Google Drive via Colab, not local disk — same as the SigLIP fine-tuning notebook. DINOv3 training doesn't use captions, so `structured_caption` isn't needed here; `product_code`/`brand`/`name` groupings are unaffected by the dataset's caption schema.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import login, whoami
login(skip_if_logged_in=False)
print(whoami())

!pip uninstall -y Pillow
!pip install --no-cache-dir --force-reinstall "Pillow==12.2.0"
!pip install -q -U "transformers>=4.56.0" accelerate safetensors huggingface_hub tqdm matplotlib


### Hugging Face access

Meta's official DINOv3 checkpoints may require accepting the model license on Hugging Face. If the model download returns an access error, accept access for the checkpoint in your HF account, then uncomment and run the cell below.

In [ ]:
# from huggingface_hub import login
# login()


## 2. Configuration, metadata, and the frozen DINOv3 encoder

In [ ]:
from pathlib import Path
from contextlib import nullcontext
import gc
import json

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from transformers import AutoImageProcessor, AutoModel


# ============================================================
# Configuration
# ============================================================

DATASET_ROOT = Path("/content/drive/MyDrive/apparel_dataset")
METADATA_PATH = DATASET_ROOT / "metadata.json"

INDEX_LABEL = "base_DINO"
INDEX_DIR = DATASET_ROOT / "embedding_indexes" / INDEX_LABEL
INDEX_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_PATH = INDEX_DIR / f"{INDEX_LABEL}_embeddings.pt"
RECORDS_PATH = INDEX_DIR / f"{INDEX_LABEL}_records.json"
CONFIG_PATH = INDEX_DIR / f"{INDEX_LABEL}_config.json"
MISSING_PATHS_PATH = INDEX_DIR / "missing_image_paths.txt"

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"

IMAGE_BATCH_SIZE = 32
NUM_WORKERS = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.float16

print("Device:", DEVICE)
print("Model:", MODEL_ID)
print("Index directory:", INDEX_DIR)


In [ ]:
# ============================================================
# Load metadata and resolve paths
# This intentionally matches the starter notebook's behavior.
# ============================================================

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)


def resolve_image_path(raw_path):
    """
    Supports paths such as:

    apparel_dataset/adidas/.../image_0.jpg
    adidas/.../image_0.jpg
    /absolute/path/image_0.jpg
    """

    raw_path = Path(raw_path)

    candidates = [
        raw_path,
        DATASET_ROOT / raw_path,
        DATASET_ROOT.parent / raw_path,
    ]

    if raw_path.parts and raw_path.parts[0] == DATASET_ROOT.name:
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[1:])
        )

    # Real files always live at DATASET_ROOT/<brand>/<slug>/<product_code>/image_N.jpg.
    # Some products still carry a stale prefix (e.g. "shoe_dataset/...") baked into
    # metadata.json from before the apparel_dataset rename -- reconstruct from the
    # last 4 path components regardless of what prefix is actually present.
    if len(raw_path.parts) >= 4:
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[-4:])
        )

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    return None


records = []
missing_paths = []

for product in metadata:
    product_code = str(product.get("product_code", "")).strip()
    image_paths = product.get("images") or []

    if not image_paths:
        continue

    for raw_path in image_paths:
        resolved_path = resolve_image_path(raw_path)

        if resolved_path is None:
            missing_paths.append(str(raw_path))
            continue

        records.append({
            "image_path": str(resolved_path),
            "product_code": product_code,
            "brand": str(product.get("brand", "")).strip(),
            "name": str(product.get("name", "")).strip(),
            "caption": str(product.get("caption", "")).strip(),
        })


if not records:
    raise RuntimeError(
        "No valid images were found. Check DATASET_ROOT and metadata.json."
    )

if missing_paths:
    MISSING_PATHS_PATH.write_text(
        "\n".join(missing_paths),
        encoding="utf-8",
    )

print(f"Products in metadata: {len(metadata):,}")
print(f"Valid images:         {len(records):,}")
print(f"Missing image paths:  {len(missing_paths):,}")


In [ ]:
# ============================================================
# Load the frozen base DINOv3 encoder
# ============================================================

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=AMP_DTYPE if USE_AMP else torch.float32,
)
model = model.to(DEVICE)
model.eval()

print("Loaded:", MODEL_ID)


In [ ]:
# ============================================================
# Image loading and embedding helpers
# ============================================================

def load_rgb_image(image_path):
    with Image.open(image_path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


def extract_global_embedding(outputs):
    """
    Prefer the model's pooled representation when available.
    Otherwise use the first/CLS token.
    """
    pooler_output = getattr(outputs, "pooler_output", None)

    if pooler_output is not None:
        features = pooler_output
    else:
        last_hidden_state = outputs.last_hidden_state
        features = last_hidden_state[:, 0]

    return F.normalize(features.float(), dim=-1)


@torch.inference_mode()
def embed_pil_images(images):
    inputs = processor(
        images=images,
        return_tensors="pt",
    )
    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    amp_context = (
        torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        )
        if USE_AMP
        else nullcontext()
    )

    with amp_context:
        outputs = model(**inputs)

    return extract_global_embedding(outputs).cpu()


## 3. Build and save the index

Run once per dataset version. Skip this section (go to §4) if you already have a saved index and just want to search or evaluate.

In [ ]:
# ============================================================
# Build and save the reusable base_DINO index
# ============================================================

all_embeddings = []

for start in tqdm(
    range(0, len(records), IMAGE_BATCH_SIZE),
    desc="Creating DINOv3 embeddings",
):
    batch_records = records[start:start + IMAGE_BATCH_SIZE]

    images = [
        load_rgb_image(record["image_path"])
        for record in batch_records
    ]

    batch_embeddings = embed_pil_images(images)
    all_embeddings.append(batch_embeddings)

embeddings = torch.cat(all_embeddings, dim=0).contiguous()

if embeddings.shape[0] != len(records):
    raise RuntimeError(
        f"Embedding count {embeddings.shape[0]} "
        f"does not match record count {len(records)}."
    )

index_payload = {
    "label": INDEX_LABEL,
    "model_id": MODEL_ID,
    "normalized": True,
    "embedding_dimension": int(embeddings.shape[1]),
    "embeddings": embeddings,
}

torch.save(index_payload, EMBEDDINGS_PATH)

with RECORDS_PATH.open("w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

config = {
    "label": INDEX_LABEL,
    "model_id": MODEL_ID,
    "dataset_root": str(DATASET_ROOT),
    "metadata_path": str(METADATA_PATH),
    "number_of_images": len(records),
    "embedding_dimension": int(embeddings.shape[1]),
    "normalized": True,
    "similarity": "cosine via normalized dot product",
}

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved embeddings:", EMBEDDINGS_PATH)
print("Saved records:   ", RECORDS_PATH)
print("Saved config:    ", CONFIG_PATH)
print("Embedding shape: ", tuple(embeddings.shape))


## 4. Load the saved index

Restart here in later sessions instead of rebuilding — requires §1-2 (setup, config, model, helper functions) to have run this session, but does not require re-running §3.

In [ ]:
payload = torch.load(EMBEDDINGS_PATH, map_location="cpu", weights_only=False)

if payload["label"] != INDEX_LABEL:
    raise ValueError(f"Expected index label {INDEX_LABEL}, found {payload['label']}.")

full_index_embeddings = payload["embeddings"].float()

with RECORDS_PATH.open("r", encoding="utf-8") as f:
    full_index_records = json.load(f)

if full_index_embeddings.shape[0] != len(full_index_records):
    raise RuntimeError("Saved embeddings and saved records have different lengths.")

print("Loaded label:", payload["label"])
print("Loaded model:", payload["model_id"])
print("Index shape:", tuple(full_index_embeddings.shape))


## 5. Nearest-neighbor search

Searches the **full** index — this is the production-search path, distinct from the held-out evaluation gallery built in §7.

In [ ]:
# ============================================================
# Exact cosine nearest-neighbor search over the full index
# ============================================================

@torch.inference_mode()
def search_base_dino_embedding(query_embedding, database_embeddings, database_records, top_k=12):
    query_embedding = F.normalize(query_embedding.float(), dim=-1)

    similarities = database_embeddings @ query_embedding
    top_k = min(top_k, len(database_records))

    scores, indices = torch.topk(similarities, k=top_k, largest=True)

    results = []
    for rank, (score, index) in enumerate(zip(scores.tolist(), indices.tolist()), start=1):
        result = dict(database_records[index])
        result.update({
            "rank": rank,
            "similarity": float(score),
            "database_index": int(index),
            "index_label": INDEX_LABEL,
        })
        results.append(result)

    return results


@torch.inference_mode()
def search_base_dino(query_image_path, database_embeddings, database_records, top_k=12):
    query_image = load_rgb_image(query_image_path)
    query_embedding = embed_pil_images([query_image])[0]

    return search_base_dino_embedding(
        query_embedding, database_embeddings, database_records, top_k=top_k,
    )


In [ ]:
# ============================================================
# Query-image selection — choose ONE method
# ============================================================

# Method A: use an image already in Google Drive.
QUERY_IMAGE_PATH = None
# Example:
# QUERY_IMAGE_PATH = "/content/drive/MyDrive/query_images/my_shoe.jpg"

# Method B: upload an image from your computer.
if QUERY_IMAGE_PATH is None:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No query image was uploaded.")

    uploaded_name = next(iter(uploaded))
    QUERY_IMAGE_PATH = f"/content/{uploaded_name}"

print("Query image:", QUERY_IMAGE_PATH)

results = search_base_dino(
    QUERY_IMAGE_PATH, full_index_embeddings, full_index_records, top_k=12,
)

for result in results:
    print(
        f"{result['rank']:>2}. "
        f"score={result['similarity']:.4f} | "
        f"product={result['product_code']} | "
        f"{result['brand']} {result['name']}"
    )


## 6. Visualize the query and nearest neighbors

In [ ]:
# ============================================================
# Visualize the query and nearest neighbors
# ============================================================

def show_search_results(query_image_path, results, columns=4):
    total_images = 1 + len(results)
    rows = int(np.ceil(total_images / columns))

    plt.figure(figsize=(4 * columns, 4 * rows))

    plt.subplot(rows, columns, 1)
    plt.imshow(load_rgb_image(query_image_path))
    plt.title("QUERY")
    plt.axis("off")

    for plot_index, result in enumerate(results, start=2):
        plt.subplot(rows, columns, plot_index)
        plt.imshow(load_rgb_image(result["image_path"]))

        title = (
            f"#{result['rank']}  {result['similarity']:.3f}\n"
            f"{result['brand']} {result['name']}\n"
            f"{result['product_code']}"
        )

        plt.title(title, fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_search_results(
    QUERY_IMAGE_PATH,
    results,
    columns=4,
)


## Current limitation

This first-stage search ranks individual catalog images. A product with many images can occupy several top positions. The next step should aggregate image-level scores by `product_code`, then rerank the top products using DINO patch descriptors and agreement across product views.

In [ ]:
# ============================================================
# Optional: aggregate image matches into product-level matches
# ============================================================

from collections import defaultdict


def aggregate_by_product(results, score_rule="max"):
    grouped = defaultdict(list)

    for result in results:
        grouped[result["product_code"]].append(result)

    product_results = []

    for product_code, matches in grouped.items():
        scores = [match["similarity"] for match in matches]

        if score_rule == "max":
            product_score = max(scores)
        elif score_rule == "mean":
            product_score = sum(scores) / len(scores)
        else:
            raise ValueError("score_rule must be 'max' or 'mean'.")

        representative = max(
            matches,
            key=lambda item: item["similarity"],
        )

        product_results.append({
            "product_code": product_code,
            "product_score": product_score,
            "best_match": representative,
            "matched_views": len(matches),
        })

    product_results.sort(
        key=lambda item: item["product_score"],
        reverse=True,
    )

    return product_results


# Search a larger image pool before aggregating.
image_results = search_base_dino(
    QUERY_IMAGE_PATH,
    full_index_embeddings,
    full_index_records,
    top_k=min(100, len(full_index_records)),
)

product_results = aggregate_by_product(
    image_results,
    score_rule="max",
)

for rank, item in enumerate(product_results[:10], start=1):
    match = item["best_match"]

    print(
        f"{rank:>2}. "
        f"score={item['product_score']:.4f} | "
        f"product={item['product_code']} | "
        f"{match['brand']} {match['name']} | "
        f"matched views={item['matched_views']}"
    )


## 7. Held-out evaluation

Builds a per-product 80/20 gallery/query split from the full index (a product's images never span both sides), then measures exact-product retrieval — R@1, R@5, R@10, MRR, median/mean rank. `gallery_*`/`heldout_query_*` are local to this section and never overwrite `full_index_*`, so the search/demo sections above always operate on the true full index.

In [ ]:
# ============================================================
# Reload full saved index, then create an in-memory 80/20 split
# ============================================================

import random
from collections import defaultdict

SPLIT_SEED = 42
GALLERY_FRACTION = 0.80

full_database_embeddings = full_index_embeddings
full_database_records = full_index_records

# Group full-index positions by product variant.
indices_by_product = defaultdict(list)

for index, record in enumerate(full_database_records):
    product_code = str(record.get("product_code", "")).strip()
    indices_by_product[product_code].append(index)


rng = random.Random(SPLIT_SEED)

gallery_indices = []
test_indices = []

for product_code, product_indices in indices_by_product.items():
    product_indices = product_indices.copy()
    rng.shuffle(product_indices)

    num_images = len(product_indices)

    # Cannot split a variant with only one image.
    # Keep it in the gallery, but exclude it from testing.
    if num_images < 2:
        gallery_indices.extend(product_indices)
        continue

    num_test = max(
        1,
        round(num_images * (1 - GALLERY_FRACTION)),
    )

    # Always leave at least one image in the gallery.
    num_test = min(num_test, num_images - 1)

    test_indices.extend(product_indices[:num_test])
    gallery_indices.extend(product_indices[num_test:])


# Stable ordering makes results reproducible.
gallery_indices = sorted(gallery_indices)
test_indices = sorted(test_indices)

gallery_index_tensor = torch.tensor(
    gallery_indices,
    dtype=torch.long,
)

test_index_tensor = torch.tensor(
    test_indices,
    dtype=torch.long,
)


# These become the active nearest-neighbor database.
gallery_embeddings = full_database_embeddings[gallery_index_tensor]
gallery_records = [
    full_database_records[index]
    for index in gallery_indices
]


# These are already encoded query images for evaluation.
heldout_query_embeddings = full_database_embeddings[test_index_tensor]
heldout_query_records = [
    full_database_records[index]
    for index in test_indices
]


print("Full index:   ", len(full_database_records))
print("Gallery 80%:  ", len(gallery_records))
print("Test 20%:     ", len(heldout_query_records))
print("Gallery shape:", tuple(gallery_embeddings.shape))
print("Test shape:   ", tuple(heldout_query_embeddings.shape))

In [ ]:
# ============================================================
# Evaluate all held-out 20% queries against the 80% gallery
# Metrics: R@1, R@5, R@10, MRR, median rank, mean rank
# ============================================================

import numpy as np
import torch

EVAL_BATCH_SIZE = 512

if len(heldout_query_records) == 0:
    raise RuntimeError("The test split is empty.")

if len(gallery_records) == 0:
    raise RuntimeError("The gallery split is empty.")

gallery_product_codes = np.array([
    str(record["product_code"]).strip()
    for record in gallery_records
])

query_product_codes = np.array([
    str(record["product_code"]).strip()
    for record in heldout_query_records
])

ranks = []

for start in range(0, len(heldout_query_embeddings), EVAL_BATCH_SIZE):
    end = min(start + EVAL_BATCH_SIZE, len(heldout_query_embeddings))

    query_batch = F.normalize(
        heldout_query_embeddings[start:end].float(),
        dim=-1,
    )

    # Shape: [number of queries, number of gallery images]
    similarity_matrix = query_batch @ gallery_embeddings.T

    sorted_indices = torch.argsort(
        similarity_matrix,
        dim=1,
        descending=True,
    ).cpu().numpy()

    for row_index, ranked_gallery_indices in enumerate(sorted_indices):
        query_code = query_product_codes[start + row_index]

        ranked_codes = gallery_product_codes[
            ranked_gallery_indices
        ]

        matching_positions = np.flatnonzero(
            ranked_codes == query_code
        )

        if len(matching_positions) == 0:
            # This should not happen if every test variant has
            # at least one corresponding gallery image.
            rank = len(gallery_records) + 1
        else:
            # Convert zero-based position to one-based rank.
            rank = int(matching_positions[0]) + 1

        ranks.append(rank)


ranks = np.asarray(ranks)

recall_at_1 = np.mean(ranks <= 1)
recall_at_5 = np.mean(ranks <= 5)
recall_at_10 = np.mean(ranks <= 10)
mrr = np.mean(1.0 / ranks)
median_rank = np.median(ranks)
mean_rank = np.mean(ranks)

print("base_DINO: held-out 20% exact-product retrieval")
print(f"Queries:     {len(heldout_query_records):,}")
print(f"Gallery:     {len(gallery_records):,}")
print(f"Candidates:  {len(set(gallery_product_codes)):,} products")
print(f"R@1:         {recall_at_1 * 100:.2f}%")
print(f"R@5:         {recall_at_5 * 100:.2f}%")
print(f"R@10:        {recall_at_10 * 100:.2f}%")
print(f"MRR:         {mrr * 100:.2f}%")
print(f"Median rank: {median_rank:.1f}")
print(f"Mean rank:   {mean_rank:.2f}")